# A closer look at Generation

<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/rag_at_a_glance.ipynb" target="_parent">RAG at a glance</a> ended with Claude's answer to
*can a king move backwards?* and a check of its citations. This notebook opens up the step in
between: what Claude is sent, where each part of it comes from, why it's written the way it is,
and what the citation check can and can't tell you.

Sections 1 to 4 need no API key.

**In this lesson**

1. **What Claude is sent**: a system prompt and one message, built from the question and the
   retrieved chunks.
2. **The system prompt**: what each rule is for, and the failure that added it.
3. **The message**: the context, the question, and the tags around them.
4. **Checking citations**: what a lookup catches, and what it misses.
5. **Asking Claude yourself**: the API key and a live call.

On Colab, run the cells from the top; the first one fetches the repo (about a minute). On your
own machine, run `uv run jupyter lab` from the repo root.

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -qq -e /content/rulebook
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import ensure_index, setup, show_table
setup()

## 1. What Claude is sent

<img src="https://ecoop.github.io/rulebook/img/flow_generation.svg" alt="The question and passages go to Claude as a prompt, and Claude generates a cited answer." width="520" style="max-width: 100%; height: auto">

A call to Claude's API takes a **system prompt**, which sets the rules for the whole exchange,
and a list of **messages**. Rulebook sends one message per question. `build_prompt` builds both
from the question and the retrieved chunks, without calling Claude, and `generate_answer` sends
exactly what it returns. The question and the retrieval are the ones from RAG at a glance:

In [ ]:
from rulebook.generate import build_prompt
from rulebook.retrieve import retrieve

ensure_index('checkers')
question = 'can a king move backwards?'
chunks = retrieve(question, domain='checkers', k=5)
system, user = build_prompt(question, chunks)
print(system)

That's the system prompt. The message holds the five chunks and the question:

In [ ]:
print(user)

Only the question was typed. The rule set's name in the system prompt came from the chunks, and
the chunks came from retrieval. The app takes the same path: its `/ask` endpoint calls
`rulebook.pipeline.ask`, which retrieves and then calls `generate_answer`.

## 2. The system prompt

The system prompt is a file, `src/rulebook/prompts/system.md`. Most of its rules were added
after something went wrong:

| Rule | What it's for | Where it came from |
|---|---|---|
| *You are an assistant that answers questions about the rules of {domains_phrase} using ONLY the excerpts provided in the &lt;context&gt; block.* | A role and a scope: these rule sets, these excerpts. `{domains_phrase}` is filled in from the chunks, which is why section 1's prompt says *Checkers*. | Until [#113](https://github.com/ecoop/rulebook/issues/113) it said *disc sports (ultimate, goaltimate)*, the app's first two rule sets. Taking the names from the chunks lets Rulebook serve any rule set without editing the prompt. |
| *Cite the source of every specific claim inline, in the form [domain rule_id]…* | Makes each claim checkable: a reader can look up the rule, and code can check the citation (section 4). | The first version. |
| *If the context does not contain the answer, say so plainly. Do NOT fill in from general knowledge.* | The basic RAG guardrail. Claude knows the rules of many games and, without this, may answer from memory. | The first version. |
| *Only the rule sets named above are available. If the question mentions any other game…* | Closes a gap the rule above left: a question that names another game. | Added in [#173](https://github.com/ecoop/rulebook/issues/173), after a live answer cited goaltimate rules that hadn't been retrieved, for someone without access to goaltimate. Its last sentence still lets an available rule set's own mention of another game through (Hearts saying it's a trick-taking game like Bridge). |
| *For questions that compare two domains, structure your answer by domain…* | Shapes answers to questions like RAG at a glance's *How do you win each game?* | The first version. |
| *When the answer depends on a term the context defines, say in plain words what the term means there, citing the definition.* | Puts a linked definition to use. Retrieval adds the definitions of terms the passages use; without this rule Claude quotes the term and moves on. | Added after [#315](https://github.com/ecoop/rulebook/pull/315): curling answers said a stone must stop beyond *the hog line at the playing end* with the playing end's definition in the context, and never said that is the far hog line. |
| *Be concise. One or two short paragraphs is usually enough.* | Length. Output tokens cost five times what input tokens do. | The first version. |

The prompt is short on purpose: each rule is something Claude may lean on too hard, so a rule
goes in when a failure shows up, not in advance.

Two settings shape the answer from outside the prompt: the model (`CLAUDE_MODEL`, now
claude-sonnet-5) and how much it reasons before answering (`CLAUDE_EFFORT`, default *low*). Low
effort cost about 17% less per answer than the model's default, with no loss against the gold
answers ([#229](https://github.com/ecoop/rulebook/issues/229)).

**Try it.** In section 1, retrieve from two rule sets instead of one: replace the `chunks = ...`
line with `chunks = retrieve_across_domains(question, ['checkers', 'tennis'], k_per_domain=3)`
(import it from `rulebook.retrieve`) and rerun that cell. The system prompt's first line now names
both.

## 3. The message

The message has three parts, in this order:

- **`<context>`**: the chunks, each under a header like `[checkers 4.2 — p.1]`, separated by
  `---`. `format_context` writes each header in the form Claude is asked to cite,
  `[domain rule_id]`, so citing a rule means copying its label. (The page number is extra; the
  citation check in section 4 ignores it.)
- **`<question>`**: the question as asked.
- **A closing reminder**: *Answer using only the context above. Cite inline with
  [domain rule_id].* It repeats the system prompt's two main rules right after the question, the
  last thing Claude reads before it writes.

**The tags.** `<context>` and `<question>` mean nothing to the API, which receives the message as
one string. They're a convention: Anthropic's
[prompting guide](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices)
recommends wrapping each kind of content in its own tag, so instructions, documents and the
question don't blur together. Any descriptive names work if they're used consistently; the
system prompt refers to *the &lt;context&gt; block* by name.

**The order.** The same guide recommends putting long documents first and the question after
them, which it reports gives better answers. Here the documents are the chunks.

## 4. Checking citations

Here is Claude's answer to the question, saved from a real run as in RAG at a glance, and the
check:

In [ ]:
import json
from rulebook.generate import find_unverified_citations

saved = json.load(open('notebooks/data/rag_at_a_glance_answers.json'))
entry = next(a for a in saved['answers'] if a['question'] == question)
print(f"Claude ({saved['model']}), recorded {saved['recorded']}:\n")
print(entry['answer'])
print('\nunverified citations:', find_unverified_citations(entry['answer'], chunks))

`find_unverified_citations` reads each `[domain rule_id]` in an answer and looks it up among the
chunks Claude was given. No model is involved; it's a lookup. Some made-up answers show what it
accepts and what it flags:

In [ ]:
examples = [
    ('a retrieved rule', 'Yes, in any direction [checkers 4.2].'),
    ('a clause inside one', 'Yes [checkers 4.2(a)].'),
    ('two rules in one bracket', 'Yes, unlike an ordinary piece [checkers 4.2, checkers 2.4].'),
    ("a rule it wasn't given", 'Yes, and it must capture when it can [checkers 3.3].'),
    ("a rule set that isn't there", 'Yes, like a chess king [chess 3.8].'),
    ('a Markdown link', 'Yes; see [the rules](https://example.com).'),
    ('the right rule, the wrong claim', 'No, only forward [checkers 4.2].'),
]
show_table(['the answer cites', 'answer', 'flagged'],
           [[kind, text, ', '.join(find_unverified_citations(text, chunks)) or '—']
            for kind, text in examples])

Two of the accepted forms were added after a correct answer was flagged:

- **A clause inside a rule** ([#226](https://github.com/ecoop/rulebook/issues/226)): chunks stop at a rule like ultimate's
  `XVII.K.2`, but its text has clauses *a*, *b* and *c*, and Claude rightly cites `XVII.K.2.a`.
- **Several rules in one bracket** ([#303](https://github.com/ecoop/rulebook/issues/303)): Claude wrote `[tennis 5.3, tennis 5.4]`
  while these saved answers were being recorded, and the check read it as one rule named
  *5.3, tennis 5.4*.

The two flagged rows are what the check is for ([#172](https://github.com/ecoop/rulebook/issues/172)): a rule Claude wasn't given,
and a rule set that isn't there at all. The app shows a note under such an answer; it doesn't
change or hide the answer.

The last row is what a lookup can't catch. *No, only forward* cites the right rule and says the
opposite of it, and the check passes it, because it only looks up the label. Catching that takes
reading: a second model call that compares each claim with the rule it cites, often called an
*LLM judge*. That's a call per answer, and Rulebook doesn't make it.

## 5. Asking Claude yourself

Everything so far ran without calling Claude. To ask Claude yourself, you need an Anthropic API
key from the [Claude Console](https://platform.claude.com/) and some API credit. The saved answer
shows what one question costs:

In [ ]:
from llm_cost_governor.pricing import usd_for_usage

usage = {'model': saved['model'], 'input_tokens': entry['input_tokens'], 'output_tokens': entry['output_tokens']}
print(f"{usage['input_tokens']} tokens in (system prompt, chunks, question), "
      f"{usage['output_tokens']} out: ${usd_for_usage(usage):.4f}")

This page leaves out the cells that call Claude. <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/generation.ipynb" target="_parent">Run it in Colab</a>
or on your own machine to use them.

**Setting the key.** On your own machine, add a line to `.env` in the repo root,
`ANTHROPIC_API_KEY=` followed by your key, and rerun the first code cell. On Colab, open
**Secrets** (the key icon on the left), add `ANTHROPIC_API_KEY` with your key as its value, and
turn on notebook access; the next cell reads it from there. Either way the key stays out of the
notebook, so it isn't saved or shared with it.

In [ ]:
from rulebook.config import settings

if not settings.anthropic_api_key and 'google.colab' in sys.modules:
    from google.colab import userdata
    settings.anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
print('API key set' if settings.anthropic_api_key else 'no API key yet: see the note above')

This sends section 1's system prompt and message, then checks the answer as in section 4.
Answers vary from run to run, so yours may differ from the saved one.

In [ ]:
from rulebook.generate import generate_answer

if not settings.anthropic_api_key:
    print('no API key yet: see Setting the key, above')
else:
    result = generate_answer(question, chunks)
    print(result.answer)
    print('\nunverified citations:', find_unverified_citations(result.answer, chunks))
    usage = {'model': settings.claude_model, 'input_tokens': result.input_tokens,
             'output_tokens': result.output_tokens}
    print(f'{result.input_tokens} tokens in, {result.output_tokens} out: ${usd_for_usage(usage):.4f}')

**Try it.** Take out the guardrails and see what Claude does without them. Copy the prompt to a
file of your own:

```
!cp src/rulebook/prompts/system.md my_prompt.md
```

Open `my_prompt.md` and delete the two rules that begin *If the context does not contain* and
*Only the rule sets named above*. Then, in a new cell, run:

```python
settings.system_prompt_file = 'my_prompt.md'
question = 'What is the offside rule in soccer?'
chunks = retrieve(question, domain='checkers', k=5)
```

and rerun the cell above. Does Claude still decline, and does it cite anything? The prompt is
read on every call; `settings.system_prompt_file = ''` goes back to the default.

## Takeaway

- What Claude is sent is built, not typed: the system prompt names the rule sets found in the
  chunks, and the message wraps the chunks and the question in tags, documents first.
- Each rule in the system prompt answers a failure: answering from memory, or citing a game that
  isn't there. Rules are added when a failure shows up, which keeps the prompt short.
- Each chunk's header is the citation Claude writes, so citations can be checked by lookup, with
  no second model call. The lookup catches a rule Claude wasn't given, but not a claim the cited
  rule doesn't support.